# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Naveed-Qasim608/Flyrank_ML_Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method choice and why

I selected a **Random Forest Classifier** for this task.

Random Forest is suitable because it can learn relationships between multiple search performance signals without requiring simple if/else rules. It also handles non-linear patterns better than a single decision tree.

The model is used as a decision-support tool to rank pages that may need content refresh. It does not predict Google's ranking algorithm or explain why rankings change.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining"] = (
    df["trend_direction"]
      .str.lower()
      .eq("down")
      .astype(int)
)

feature_cols = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining"]

print("Features:", feature_cols)
print("Samples:", len(X))

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split design

The dataset is divided into training and testing sets.

The training data is used to learn patterns, while the testing data evaluates performance on unseen pages.

Using separate training and testing data provides a more honest estimate of model performance than evaluating on the same data used for training.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## Train and compare with baseline

The baseline predicts pages using a simple ranking based on impressions.

The Random Forest model is trained using multiple observed signals.

Both approaches are evaluated on the same test data so the comparison is fair.

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier
import pandas as pd

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

baseline_acc = accuracy_score(
    y_test,
    baseline.predict(X_test)
)

rf_acc = accuracy_score(
    y_test,
    rf.predict(X_test)
)

comparison = pd.DataFrame({
    "Model": [
        "Baseline",
        "Random Forest"
    ],
    "Accuracy": [
        baseline_acc,
        rf_acc
    ]
})

comparison

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Errors and interpretation

The model correctly identifies many declining pages but still makes mistakes.

Some pages have similar search signals even though their outcomes differ, making them difficult to classify correctly.

The model appears to rely mainly on impressions, ranking position, and content freshness. These observed relationships support prioritization decisions but should not be interpreted as causal explanations.

In [ ]:
from sklearn.metrics import classification_report

pred = rf.predict(X_test)

print(classification_report(
    y_test,
    pred,
    digits=3
))

importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

importance

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.